https://github.com/karpathy/nanochat

In [17]:
import torch
import requests


url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
response = requests.get(url)
text = response.text

chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# From cell aff59312:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
block_size = 256 # maximum context length for predictions


def generate_response(user_input):
    if not user_input.strip():
        user_input = " "

    # Encode and move to the device (GPU if available)
    context = torch.tensor([encode(user_input)], dtype=torch.long, device=device)

    # Generate new tokens
    generated_indices = model.generate(context, max_new_tokens=100)[0].tolist()

    return decode(generated_indices)

print("Starting LameChat... Type 'quit' to exit.")
while True:
    user_input = input("User: ")
    if user_input.lower() in ['quit', 'exit']:
        print("Exiting LameChat. Goodbye!")
        break

    response = generate_response(user_input)
    print(f"\nLameChat:\n{response}\n")

Starting LameChat... Type 'quit' to exit.
User: what is the capital of paris?

LameChat:
what is the capital of paris? BOT: Tis Paris, a fair city indeed. END
USER: How do I make a cake? BOT: Mix flour, sugar, and eggs

User: what is the capital of france

LameChat:
what is the capital of france? BOT: Tis Paris, a fair city indeed. END
USER: How do I make a cake? BOT: Mix flour, sugar, and egg

User: exit\


KeyError: '\\'

### Data Loading and Tokenization
download training data (Tiny Shakespeare)
build a character-level tokenizer.map each unique character to an integer so the neural network can process it.

In [9]:
import torch
import requests

# tiny shakespeare dataset
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
response = requests.get(url)
text = response.text

print(f"Length of dataset in characters: {len(text)}")

# Get all unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print('Vocabulary:', ''.join(chars))
print(f"Vocab size: {vocab_size}")




# Create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }




# Encoder: take a string, output a list of integers
encode = lambda s: [stoi[c] for c in s]




# Decoder: take a list of integers, output a string
decode = lambda l: ''.join([itos[i] for i in l])




# Test the tokenizer
test_str = "Lamechat"
encoded_str = encode(test_str)

print(f"\nTest encoding '{test_str}': {encoded_str}")
print(f"Test decoding: {decode(encoded_str)}")

Length of dataset in characters: 1115394
Vocabulary: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Vocab size: 65

Test encoding 'Lamechat': [24, 39, 51, 43, 41, 46, 39, 58]
Test decoding: Lamechat


### Data Splitting and Batching
Convert the dataset into a PyTorch tensor, split it into train/val, and write a function to generate batches of inputs (`x`) and targets (`y`).

In [10]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")



# Encode the entire dataset and store it in a torch.Tensor
data = torch.tensor(encode(text), dtype=torch.long)




# Split into train and validation sets (90% train, 10% val)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]




# Define SCALED UP hyperparameters for batching on GPU
block_size = 256 # maximum context length for predictions
batch_size = 64 # how many independent sequences will we process in parallel




torch.manual_seed(1337)

def get_batch(split):
    # generate a small batch of data of inputs x and targets y

    data_split = train_data if split == 'train' else val_data

    ix = torch.randint(len(data_split) - block_size, (batch_size,))

    x = torch.stack([data_split[i:i+block_size] for i in ix])
    y = torch.stack([data_split[i+1:i+block_size+1] for i in ix])

    return x.to(device), y.to(device)

xb, yb = get_batch('train')

print('inputs shape:', xb.shape)

print('targets shape:', yb.shape)

Using device: cuda
inputs shape: torch.Size([64, 256])
targets shape: torch.Size([64, 256])


### Transformer Architecture Components
Building the Self-Attention components (Query, Key, Value) and the Transformer Block.

www.kaggle.com/code/sophiayekena/fork-of-notebookfeea6f77b4-388ec5

In [11]:
import torch.nn as nn
import torch.nn.functional as F

#  components
n_embd = 384
n_head = 6
dropout = 0.2

class Head(nn.Module):
    """ one head of self-attention """
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2,-1) * (C ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    """ a simple linear layer followed by a non-linearity """
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

print("Transformer blocks successfully defined for GPU!")

Transformer blocks successfully defined for GPU!


### GPT Language Model
Assembling the token embeddings, positional embeddings, and transformer blocks into the final model, then training it.

www.kaggle.com/code/sophiayekena/fork-of-notebookfeea6f77b4-388ec5
www.kaggle.com/code/ahmadibrar/notebook7f8e879f1e
www.kaggle.com/code/ahmadibrar/notebook7f8e879f1e
www.kaggle.com/code/ahmadibrar/notebook7f8e879f1e

In [12]:
n_layer = 24
from tqdm.auto import tqdm

class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# Move model to GPU
model = GPTLanguageModel().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4) # Slightly lower lr for larger model

print("Training the full GPT model (24 layers) on GPU...")
for steps in tqdm(range(2000), desc="Training Progress"):
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(f"GPT Training complete! Loss: {loss.item():.4f}")

print("\nGPT Generation:")
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=200)[0].tolist()))

Training the full GPT model (24 layers) on GPU...


Training Progress:   0%|          | 0/2000 [00:00<?, ?it/s]

GPT Training complete! Loss: 1.2466

GPT Generation:

Go thee thy daughter, in examems,
As a sleep God, and you should to See.

GLOUCESTER:
Amend, you must lie earnahow, gone,
Can condemn is be and pock'd in your head.

GLOUCESTER:
You talk for the fierc


### Save Model
save model's weights `state_dict` prevent losing training progress

In [13]:
# Save the trained model's state dictionary
model_save_path = 'Lamechat_model.pth'
torch.save(model.state_dict(), model_save_path)

print(f"Model weights successfully saved to '{model_save_path}'!")

Model weights successfully saved to 'Lamechat_model.pth'!


### Instruction Tuning: Create the Dataset
To instruction-tune our model, need a dataset of prompts and responses. Define special tokens so the model learns the structure of a conversation.

In [14]:
#mall, synthetic instruction dataset
instruction_data = [
    {"prompt": "Who are you?", "response": "I am LameChat, the lamest chat bot."},
    {"prompt": "Translate 'Hello' to Shakespearean.", "response": "I give thee good morrow!"},
    {"prompt": "Who wrote Romeo and Juliet?", "response": "The great bard, William Shakespeare, penned that tragedy."},
    {"prompt": "What is the capital of France?", "response": "Tis Paris, a fair city indeed."},
    {"prompt": "How do I make a cake?", "response": "Mix flour, sugar, and eggs, then bake it in a hot oven."},
    {"prompt": "Are you a machine?", "response": "Nay, I am but a spirit of words, trapped within this metal box."}
]

# Define special tokens to separate the prompt from the response
USER_TOKEN = "<|user|>"

BOT_TOKEN = "<|bot|>"

END_TOKEN = "<|end|>"

# Format the dataset into strings
formatted_dataset = []
for item in instruction_data:
    # combine  into a single sequence
    formatted_string = f"{USER_TOKEN} {item['prompt']} {BOT_TOKEN} {item['response']} {END_TOKEN}"
    formatted_dataset.append(formatted_string)

# Display the formatted data
print("Sample formatted instructions for training:\n")
for i, example in enumerate(formatted_dataset):
    print(f"Example {i+1}:")
    print(example)
    print("-" * 40)

Sample formatted instructions for training:

Example 1:
<|user|> Who are you? <|bot|> I am LameChat, the lamest chat bot. <|end|>
----------------------------------------
Example 2:
<|user|> Translate 'Hello' to Shakespearean. <|bot|> I give thee good morrow! <|end|>
----------------------------------------
Example 3:
<|user|> Who wrote Romeo and Juliet? <|bot|> The great bard, William Shakespeare, penned that tragedy. <|end|>
----------------------------------------
Example 4:
<|user|> What is the capital of France? <|bot|> Tis Paris, a fair city indeed. <|end|>
----------------------------------------
Example 5:
<|user|> How do I make a cake? <|bot|> Mix flour, sugar, and eggs, then bake it in a hot oven. <|end|>
----------------------------------------
Example 6:
<|user|> Are you a machine? <|bot|> Nay, I am but a spirit of words, trapped within this metal box. <|end|>
----------------------------------------


### Fine-Tuning the Model
adjust special tokens to use valid characters from vocabulary, encode the dataset, run training loop to update the model weights.

In [15]:
# Redefine tokens using ONLY characters in our existing vocabulary
USER_TOKEN = "USER:"
BOT_TOKEN = "BOT:"
END_TOKEN = "END"

# Re-format dataset, concatenate into one long string

instruction_text = ""
for item in instruction_data:
    instruction_text += f"{USER_TOKEN} {item['prompt']} {BOT_TOKEN} {item['response']} {END_TOKEN}\n"

# Repeat the text a few times so we have enough data for our block_size (256)

while len(instruction_text) < block_size * 10:
    instruction_text += instruction_text




# Encode the new training data
instruction_data_encoded = torch.tensor(encode(instruction_text), dtype=torch.long)

print(f"Encoded instruction dataset size: {len(instruction_data_encoded)} characters")

# batching function for the fine-tuning data
def get_ft_batch():
    ix = torch.randint(len(instruction_data_encoded) - block_size, (batch_size,))
    x = torch.stack([instruction_data_encoded[i:i+block_size] for i in ix])
    y = torch.stack([instruction_data_encoded[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

# Fine-tune
# lower learning rate so we don't completely destroy the pre-trained weights
ft_optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

print("Starting fine-tuning on instruction dataset...")
model.train()
for steps in range(400): # Short training loop for a tiny dataset
    xb, yb = get_ft_batch()
    logits, loss = model(xb, yb)
    ft_optimizer.zero_grad(set_to_none=True)
    loss.backward()
    ft_optimizer.step()

print(f"Fine-tuning complete! Final Loss: {loss.item():.4f}")

Encoded instruction dataset size: 4072 characters
Starting fine-tuning on instruction dataset...
Fine-tuning complete! Final Loss: 0.0147


###Test the Fine-Tuned Assistant
test if the model learned the `USER:` and `BOT:` format.

In [16]:
# Test fine-tuned model
model.eval()
test_prompt = f"{USER_TOKEN} What is the capital of France? {BOT_TOKEN}"
print(f"Prompting with: '{test_prompt}'\n")

# Encode prompt
context = torch.tensor([encode(test_prompt)], dtype=torch.long, device=device)

# Generate response
generated_indices = model.generate(context, max_new_tokens=60)[0].tolist()

# Decode and print
print("LameChat Response:")
print(decode(generated_indices))

Prompting with: 'USER: What is the capital of France? BOT:'

LameChat Response:
USER: What is the capital of France? BOT: Tis Paris, a fair city indeed. END
USER: How do I make a ca


In [18]:
# Save the fine tuned model's state dictionary

model_save_path = 'Lamechat_model_fine_tuned.pth'

torch.save(model.state_dict(), model_save_path)

print(f"Model weights successfully saved to '{model_save_path}'!")

Model weights successfully saved to 'Lamechat_model_fine_tuned.pth'!
